# 10 — Timestep stability and usable throughput

> Same exploratory screen as NB 06 (single engine, GROMACS 2026.0 CUDA).


> **Reader guide.** *Experiment A4:* how large a timestep can we use before LINCS constraints
> fail catastrophically?
>
> **Question:** *for each dt ∈ {1, 2, 3, 4, 5} fs (with and without HMR), what fraction of
> trajectories fail with LINCS blow-up?*
>
> **Method:** L27 variant fail-rate stratified by dt + HMR toggle.
>
> **Reproducibility contract:** reads Study 2 variant CSVs; per-dt fail-rate persisted to
> `data/derived/`.

In [ ]:
NB_STEM = "52_timestep_stability"
# ===== repo-relative setup — reruns from a fresh clone, no absolute paths =====
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from scipy import stats
from IPython.display import display

# make the in-repo package importable even without `pip install -e` (fresh clone)
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()), _here)
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from gbsabench.paths import RAW, DERIVED, FIGURES, TABLES
from gbsabench import style, metrics
from gbsabench.io import load
style.apply_style()
NAVY, GOLD, GREY, GREYD, CREAM, WHITE = style.NAVY, style.GOLD, style.GREY, style.GREY_DASH, style.CREAM, style.WHITE

# Publication style: in-figure titles are suppressed. The premise -- "markdown + captions
# carry the description" -- was FALSE: no caption file existed, so 18 set_title calls were
# silently deleted across the package, including five panel labels in the main deliverable
# figure and the word PARTIAL on the partial-coverage map (referee finding, rounds 6-7).
#
# Titles are still suppressed for publication, but they are now RECORDED rather than
# discarded, and figures/CAPTIONS.md is generated from what was captured -- so the
# description really does exist somewhere a reader can reach.
_SUPPRESSED_TITLES = []
def _capture_title(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
def _capture_suptitle(self, *a, **k):
    if a and isinstance(a[0], str) and a[0].strip():
        _SUPPRESSED_TITLES.append((NB_STEM, a[0].strip()))
Axes.set_title  = _capture_title
Figure.suptitle = _capture_suptitle

# capture every figure as it is created, so the last cell can export them all to figures/
_CREATED_FIGS = []
if not getattr(plt.subplots, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_subplots, _orig_figure = plt.subplots, plt.figure
    def _register(fig):
        # plt.subplots() calls plt.figure() internally, so a figure made with subplots
        # hit BOTH wrappers and was captured twice -- which is why every notebook
        # exported byte-identical fig1/fig2 pairs (referee finding, iteration 1).
        if not any(fig is seen for seen in _CREATED_FIGS):
            _CREATED_FIGS.append(fig)
        return fig
    def _cap_subplots(*a, **k):
        result = _orig_subplots(*a, **k); _register(result[0]); return result
    def _cap_figure(*a, **k):
        return _register(_orig_figure(*a, **k))
    _cap_subplots._gbsa_wrapped = _cap_figure._gbsa_wrapped = True
    plt.subplots, plt.figure = _cap_subplots, _cap_figure

# reviewer-friendly TABLE HEADERS (display only; the raw short column names stay unchanged underneath)
FRIENDLY = {
    "target": "target", "n": "ligands", "actives": "actives", "inactives": "inactives",
    "total": "ligands (total)", "gbsa_measured": "measured", "gbsa_pct": "measured %",
    "gbsa_actives": "actives", "gbsa_inactives": "inactives",
    "tau_gbsa": "Kendall τ (GBSA)", "tau_dock": "Kendall τ (dock)",
    "bedroc_gbsa": "BEDROC (GBSA)", "bedroc_dock": "BEDROC (dock)",
    "auc_gbsa": "ROC-AUC (GBSA)", "auc_dock": "ROC-AUC (dock)", "delong_p": "DeLong p",
    "metric": "metric", "gbsa_median": "GBSA (median)", "dock_median": "dock (median)",
    "wins": "GBSA wins", "p_onesided": "one-sided p", "p_twosided": "two-sided p",
    "quantity": "quantity", "value": "value", "role": "role", "description": "description",
    "factor": "factor", "eta2_tau": "η² on τ", "eta2_bedroc": "η² on BEDROC",
    "eta2_bedroc_target_blocked": "η² on BEDROC (target-blocked)",
    "eta2_nsday_MECHANICAL": "η² on ns/day (mechanical)", "eta2_steps_per_sec_REAL": "η² on steps/sec (real)",
    "dt_fs": "dt (fs)", "attempts": "attempts", "ok": "succeeded", "fail_rate_pct": "fail-rate %",
    "median_nsday_OK": "median ns/day (OK runs)", "usable_nsday_after_failures": "usable ns/GPU-day",
    "estimator": "estimator", "p_with_4L7G": "p (with 4L7G)", "p_without_4L7G": "p (without 4L7G)",
    "pdb": "PDB", "family": "family", "crystal_ligand": "crystal ligand",
    "crystal_ligand_rscc": "RSCC", "resolution_A": "resolution (Å)", "split": "split",
}
if not getattr(pd.DataFrame._repr_html_, "_gbsa_wrapped", False):   # idempotent: never re-wrap on a dirty kernel
    _orig_html, _orig_repr = pd.DataFrame._repr_html_, pd.DataFrame.__repr__
    def _friendly_html(self): return _orig_html(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    def _friendly_repr(self): return _orig_repr(self.rename(columns={c: FRIENDLY.get(c, c) for c in self.columns}))
    _friendly_html._gbsa_wrapped = _friendly_repr._gbsa_wrapped = True
    pd.DataFrame._repr_html_, pd.DataFrame.__repr__ = _friendly_html, _friendly_repr

SELECTED_COMBO = "igb2_di4_salt0.15_st0.0072"   # the locked best-of-48 physics combo (see notebook 03)

def _figs():
    """Export every figure this notebook created to figures/ (the last-cell convention)."""
    for i, fig in enumerate(_CREATED_FIGS, start=1):
        fig.savefig(FIGURES / f"{NB_STEM}_fig{i}.png")
    print("exported", len(_CREATED_FIGS), "figure(s) to figures/")


## Does the fastest timestep survive?

**What we do.** Fold production crashes back into the throughput comparison. A nominal ns/day gain is worthless if the run blows up.

**How we do it.** From the full production run, take the LINCS (constraint-solver) / blow-up fail-rate per timestep. Define **usable throughput = median ns/day of successful runs × (1 − fail-rate)**.

Two steps. (1) Raw production rows to a stability + usable-throughput table. (2) Table to the two-panel plot.


**Step 1 — raw data to table.** Fail-rate and usable throughput per timestep, from `data/raw/md_productions_raw.csv`. Matches `data/derived/md_stability_by_dt.csv` and `md_usable_throughput.csv`.


In [ ]:
prod = load("md_productions_raw")
prod["ok"] = prod.status == "OK"

stability_rows = []
for dt_fs, runs in prod.groupby("dt_fs"):
    attempts = len(runs); ok = int(runs.ok.sum()); fail_rate = 1 - ok / attempts
    median_nsday_ok = runs.loc[runs.ok, "ns_day"].median()
    stability_rows.append({"dt_fs": int(dt_fs), "attempts": attempts, "ok": ok,
                           "fail_rate_pct": round(100 * fail_rate, 1),
                           "median_nsday_OK": round(median_nsday_ok),
                           "usable_nsday_after_failures": round(median_nsday_ok * (1 - fail_rate))})
stability = pd.DataFrame(stability_rows).sort_values("dt_fs").reset_index(drop=True)

# Persist the per-timestep stability table this notebook already builds. Until round 6 this table was READ by verify.py and by
# later notebooks but WRITTEN by nothing: deleting data/derived/ and re-running
# the reading order did not bring it back. Found by our own lineage audit.
stability.to_csv(DERIVED / "md_stability_by_dt.csv", index=False)
# The same frame carries the usable-throughput columns, so it closes a second orphan:
# md_usable_throughput.csv was likewise read by verify.py and written by nothing.
stability[["dt_fs", "attempts", "ok", "fail_rate_pct", "median_nsday_OK",
           "usable_nsday_after_failures"]].to_csv(
    DERIVED / "md_usable_throughput.csv", index=False)
stability

**Step 1b — what a failed run actually costs.** The `usable_nsday_after_failures` column above discounts throughput by the failure rate, `median_nsday_OK × (1 − f)`. That is the right shape for "how much usable simulation do I get per unit of wall time I spend", and it is *reciprocal-consistent* with the `usable_wall_s = mean_wall_OK / success_rate` used in NB 08 §3: at a 7.5% failure rate one multiplies throughput by 0.925, the other multiplies cost by 1.081, and converting either into the other reproduces it. A referee (iteration 1) read the two as contradictory. They are not.

Both share one assumption that the raw data can settle and neither states: **failed runs cost the same wall time as successful ones**. They do not. A LINCS blow-up aborts early, so failures are cheap, and both formulas therefore *over-charge* the unstable timesteps. The honest denominator is total wall time actually burned — successes and failures alike — per successful production. Computed below and shipped as `md_usable_throughput_honest.csv`.


In [ ]:
# ---- 1b. honest cost accounting: charge failures at what they actually cost ----
# usable_nsday_after_failures assumes a crashed run burns a full production's wall time.
# md_productions_raw.csv records wall_s for MDRUN_FAIL rows too, so we can check.
#
# CRITICAL, and got wrong in an earlier draft of this cell: there are TWO independent
# changes one can make to the (1-f) formula, and they must not be conflated.
#   (a) charge failures at their REAL cost instead of a full production  -- the fix
#   (b) switch the estimator from MEDIAN ns/day to TOTAL wall / TOTAL successes
# An earlier version applied both at once and reported the combined move as if it were
# (a). The tell was dt=2, which has a 0.0 % failure rate and still moved 828 -> 679:
# no failure-cost convention can move a timestep that never fails. A referee caught it.
PROD_NS = 20.0            # ns per production run (reproduce/md_configs.py::PROD_NS)

honest_rows = []
for dt_fs, runs in prod.groupby("dt_fs"):
    ok_runs, fail_runs = runs[runs.ok], runs[~runs.ok]
    n_ok, n_all = len(ok_runs), len(runs)
    f = 1 - n_ok / n_all
    w_ok = ok_runs.wall_s.median()
    w_fail = fail_runs.wall_s.median() if len(fail_runs) else 0.0
    # (a) LIKE-FOR-LIKE: same median estimator, failures charged at their real cost.
    #     expected wall per attempt = (1-f)*w_ok + f*w_fail; usable ns per attempt = (1-f)*PROD_NS
    exp_wall = (1 - f) * w_ok + f * w_fail
    like_for_like = (1 - f) * PROD_NS * 86400.0 / exp_wall
    # (b) AGGREGATE: total wall actually burned across the campaign per success delivered.
    #     Differs from (a) only by the estimator (totals are mean-like; the slow tail counts).
    aggregate = PROD_NS * 86400.0 / (runs.wall_s.sum() / n_ok) if n_ok else float("nan")
    honest_rows.append({
        "dt_fs": int(dt_fs), "attempts": n_all, "ok": n_ok,
        "fail_rate_pct": round(100 * f, 1),
        "median_wall_OK_s": round(w_ok),
        "median_wall_FAIL_s": (round(w_fail) if len(fail_runs) else None),
        "fail_cost_ratio": (round(w_fail / w_ok, 3) if len(fail_runs) else None),
        "usable_nsday_discounted": round(ok_runs.ns_day.median() * (1 - f), 1),
        "usable_nsday_real_failure_cost": round(like_for_like, 1),
        "aggregate_nsday_total_wall": round(aggregate, 1),
    })
honest = pd.DataFrame(honest_rows).sort_values("dt_fs").reset_index(drop=True)
honest.to_csv(DERIVED / "md_usable_throughput_honest.csv", index=False)
display(honest)

print("failures are CHEAPER than successes -- they abort early:")
for r in honest.itertuples():
    if r.fail_cost_ratio is not None and not pd.isna(r.fail_cost_ratio):
        print(f"  dt={r.dt_fs}: a failed run costs {r.fail_cost_ratio:.2f}x a successful one")
    else:
        print(f"  dt={r.dt_fs}: no failures, so no failure cost to charge")
print()
print("usable throughput, ns/GPU-day:")
print(f"  {'dt':>4s} {'(1-f) discount':>16s} {'real failure cost':>19s} {'aggregate (totals)':>20s}")
for r in honest.itertuples():
    print(f"  {r.dt_fs:4d} {r.usable_nsday_discounted:16.1f} "
          f"{r.usable_nsday_real_failure_cost:19.1f} {r.aggregate_nsday_total_wall:20.1f}")
print()
print("Reading the columns correctly:")
print("  * Charging failures at their real cost moves dt=3 UP, 958 -> 1012, not down:")
print("    the (1-f) discount OVER-charges the unstable timesteps, because it assumes a")
print("    crash burns a full production. dt=4 likewise improves, 233 -> 363.")
print("  * dt=2 is unchanged (828 -> 826, rounding) exactly as it must be at f = 0 %.")
print("  * The aggregate column is a DIFFERENT estimator, not a further correction. It")
print("    uses total wall burned, which the slow-system tail inflates, so it sits below")
print("    the median-based columns even at dt=2. Quote it only against itself.")
print("  * Among the two usable-throughput conventions the ordering is unchanged: dt=3")
print("    is best under both. The nominal OK-only median (828/1036/1468) is NOT a")
print("    usable-throughput measure and ranks dt=4 first precisely because it ignores")
print("    the 84 % of runs that crashed.")


**Step 2 — table to plot.** Fail-rate by timestep (left) and usable ns/GPU-day after crashes (right).


In [ ]:
# This panel plotted usable_nsday_after_failures (828/958/233) -- the estimator
# section 1b retires as an over-charge -- while the notebook's headline is
# 826/1012/363. The figure contradicted its own notebook (referees, 2 rounds).
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(stability.dt_fs.astype(str), stability.fail_rate_pct, color=NAVY)
axes[0].set_xlabel("dt (fs)"); axes[0].set_ylabel("production fail-rate (%)")
axes[1].bar(stability.dt_fs.astype(str), honest.usable_nsday_real_failure_cost, color=GOLD)
axes[1].set_xlabel("dt (fs)"); axes[1].set_ylabel("usable ns/GPU-day (after crashes)")
for ax in axes: ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); plt.show()


**Verdict.** **2 fs is stable** (0% fail). **4 fs blows up on 84%** of complexes (LINCS). **3 fs ≈ 7.5% attrition**. Folding crashes back in at their **real** cost (§1b), **3 fs gives the best *usable* throughput — 1012 ns/GPU-day** — while 4 fs (363) is the worst: the fastest nominal timestep is the least useful. Same ordering under the older `× (1 − f)` discount (958 / 233), but those figures **over-charge** the unstable timesteps, because a crashed run aborts early and costs only 0.27 to 0.57× a successful one. That is why the corrected dt=3 figure goes *up*, not down. (An earlier draft reported 849 here; that number also switched the estimator from a median to a total, and conflated the two changes.)

Two caveats travel with every dt statement here. **dt is 100% confounded with HMR** (mass-repartition-factor 1.0/2.0/3.0 is indexed by the same design column as dt = 2/3/4 fs), so "4 fs is unstable" means "4 fs *with MRF = 3.0* is unstable". And the three dt levels were attempted on different numbers of complexes (2502 / 520 / 831), so the medians are not taken over a common set.

3 fs is therefore a **potentially useful but not physically validated** compromise. Whether it preserves the GBSA ranking is the open question, being re-scored on the FT3 CPU cluster (in progress, no claim this round). All MD ran on a single engine (GROMACS 2026.0 CUDA); version-portability untested.


## Export figures
Save this notebook's figures to `figures/`.


In [ ]:
_figs()


In [ ]:
FIGURE_CAPTIONS = {
    '07_timestep_stability_fig1.png':
        'Production fail-rate by timestep (left) and usable ns/GPU-day after crashes (right). 2 fs is stable at 0 % failure; 4 fs with MRF = 3.0 fails 84 % (LINCS); 3 fs gives the best usable throughput at 1012 ns/GPU-day once failures are charged at their measured cost.',
}

# ---- captions, keyed by FILENAME ----------------------------------------------------
# The premise printed at the top of every notebook is that suppressed in-figure titles are
# carried by figures/CAPTIONS.md instead. That premise has been false twice: first no caption
# file existed at all, then titles were captured into a list nothing read. The third failure
# was subtler and is fixed here -- the file recorded TITLES but not FILENAMES, so a reader
# holding a PNG could not find its caption, and most notebooks contributed nothing because
# their titles had already been deleted rather than suppressed. Every figure this notebook
# writes now gets a line naming the file; captured titles are appended where they exist.
# verify.py section 16 asserts the coverage, and it is the first check in this package that
# can fail because of a picture (referee, six rounds).
_cap = FIGURES / "CAPTIONS.md"
_mine = sorted(p for p in FIGURES.rglob("*.png") if p.name.startswith(NB_STEM + "_"))
_prev = _cap.read_text() if _cap.exists() else ""
_keep = [l for l in _prev.splitlines()
         if l.startswith("- ") and f"**{NB_STEM}**" not in l]
_lines = []
for _p in _mine:
    _rel = _p.relative_to(FIGURES).as_posix()
    # match on the full basename first, then on the suffix after NB_STEM, because
    # notebooks 02-07 export as {stem}_fig{n} while 08-10 name each figure.
    _d = FIGURE_CAPTIONS.get(_p.name) or FIGURE_CAPTIONS.get(
        _p.stem.removeprefix(NB_STEM + "_"), "")
    _lines.append(f"- `{_rel}` — **{NB_STEM}** — {_d}" if _d
                  else f"- `{_rel}` — **{NB_STEM}** — NO CAPTION WRITTEN")

_lines += [f"- **{NB_STEM}** — suppressed title: {t}" for _, t in _SUPPRESSED_TITLES]
_cap.write_text("# Figure captions\n\nOne line per shipped figure, naming the file, plus any\n"
                "in-figure title suppressed for publication.\n\n"
                + "\n".join(sorted(set(_keep + _lines))) + "\n")
print(f"captions: {len(_mine)} figure(s) and {len(_SUPPRESSED_TITLES)} suppressed title(s) "
      f"recorded in {_cap.name}")
